In [ ]:
import sys
!{sys.executable} -m pip install -q pandas==2.2.3 requests==2.32.3 beautifulsoup4==4.12.3 lxml==5.3.0 rapidfuzz==3.9.7 tqdm==4.66.5

In [ ]:
import os,re,time,random,hashlib,unicodedata
from datetime import datetime, timezone
import pandas as pd
from rapidfuzz import fuzz

def remove_accents(text):
    text='' if text is None else str(text)
    return ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))
def normalize_text(text):
    t=remove_accents(text).lower(); t=re.sub(r'[^a-z0-9\s]',' ',t)
    return re.sub(r'\s+',' ',t).strip()
def normalize_name(name): return normalize_text(name)
def stable_hash(text): return hashlib.sha256(str(text).encode('utf-8')).hexdigest()[:16]
def score_name_match(a,b): return float(fuzz.token_sort_ratio(normalize_name(a),normalize_name(b)))
def classify_match(score):
    return 'exact_or_very_strong_name' if score>=95 else 'strong_candidate' if score>=88 else 'medium_candidate' if score>=75 else 'weak_candidate' if score>=60 else 'discard'
def now_utc(): return datetime.now(timezone.utc).isoformat()
def sha256_file(path):
    if not path or not os.path.exists(path): return ''
    h=hashlib.sha256();
    with open(path,'rb') as f:
        for ch in iter(lambda:f.read(8192),b''): h.update(ch)
    return h.hexdigest()
def save_csv(df,path): os.makedirs(os.path.dirname(path),exist_ok=True); df.to_csv(path,index=False,encoding='utf-8-sig')
EVID_COLS=['seed_id', 'nombre_persona_input', 'normalized_name_input', 'estado_input', 'puesto_input', 'dependencia_input', 'notas_input', 'source', 'source_type', 'source_country', 'source_official', 'source_category', 'source_reliability', 'matched_record_name', 'matched_record_normalized_name', 'matched_alias', 'matched_entity_type', 'matched_role', 'matched_position', 'matched_agency', 'matched_state', 'matched_country', 'matched_identifier', 'matched_company', 'match_score_pct', 'match_strength', 'match_method', 'matched_fields', 'conflicting_fields', 'requires_review', 'identity_confidence_pct', 'signal_type', 'signal_category', 'severity', 'risk_layer', 'risk_level_candidate', 'confidence_pct', 'evidence_title', 'evidence_summary', 'evidence_snippet', 'evidence_keywords', 'evidence_date', 'evidence_url', 'source_url', 'search_query', 'search_engine', 'retrieved_at', 'raw_file_path', 'raw_file_sha256', 'involvement_summary', 'explanation', 'limitations', 'recommended_action']

In [ ]:

import requests, xml.etree.ElementTree as ET
from tqdm import tqdm
FORCE_DOWNLOAD=False
os.makedirs('notebooks/raw/ofac',exist_ok=True); os.makedirs('notebooks/raw/un',exist_ok=True)
peps=pd.read_csv('notebooks/output/00_peps_normalized.csv',dtype=str).fillna('') if os.path.exists('notebooks/output/00_peps_normalized.csv') else pd.DataFrame([{'seed_id':'fallback','nombre_persona_input':'Nombre Ejemplo','normalized_name_input':'nombre ejemplo','estado_input':'','puesto_input':'','dependencia_input':'','notas_input':'','has_only_name':True}])
bench=[]; rows=[]
def download(url,path):
    if os.path.exists(path) and not FORCE_DOWNLOAD: return True,''
    try:
        r=requests.get(url,timeout=45)
        if r.status_code in [401,403]: return False,f'blocked_http_{r.status_code}'
        r.raise_for_status(); open(path,'wb').write(r.content); return True,''
    except Exception as e: return False,str(e)
def add(p,src,name,score,raw,src_url):
    rows.append({**{c:'' for c in EVID_COLS},'seed_id':p['seed_id'],'nombre_persona_input':p['nombre_persona_input'],'normalized_name_input':p['normalized_name_input'],'estado_input':p.get('estado_input',''),'puesto_input':p.get('puesto_input',''),'dependencia_input':p.get('dependencia_input',''),'notas_input':p.get('notas_input',''),'source':src,'source_type':'sanctions_list','source_country':'US' if src=='OFAC' else 'UN','source_official':True,'source_category':'sanctions','source_reliability':'official','matched_record_name':name,'matched_record_normalized_name':normalize_name(name),'match_score_pct':score,'match_strength':classify_match(score),'match_method':'rapidfuzz_token_sort','requires_review':True,'identity_confidence_pct':min(score,75),'signal_type':'sanctions_candidate','signal_category':'official_sanctions','severity':'high','risk_layer':'sanctions','risk_level_candidate':'critical_candidate' if score>=95 else 'high_review','confidence_pct':min(score,75),'evidence_title':'Coincidencia candidata en lista de sanciones','evidence_summary':'Coincidencia nominal en fuente oficial; requiere revisión por falta de identificadores adicionales.','retrieved_at':now_utc(),'raw_file_path':raw,'raw_file_sha256':sha256_file(raw),'source_url':src_url,'involvement_summary':'Coincidencia nominal en fuente oficial; requiere revisión por falta de identificadores adicionales.','limitations':'Solo nombre como identificador','recommended_action':'verify_identity'})

sources=[('OFAC','https://sanctionslistservice.ofac.treas.gov/api/PublicationPreview/exports/SDN.CSV','notebooks/raw/ofac/sdn.csv'),('UN','https://scsanctions.un.org/resources/xml/en/consolidated.xml','notebooks/raw/un/consolidated.xml')]
for src,url,path in sources:
    t0=time.perf_counter(); ok,err=download(url,path); names=[]
    if ok:
        try:
            if src=='OFAC':
                df=pd.read_csv(path,dtype=str).fillna('')
                col=df.columns[1] if len(df.columns)>1 else df.columns[0]
                names=df[col].astype(str).tolist()
            else:
                root=ET.parse(path).getroot()
                for p in root.iter():
                    if p.tag.upper().endswith('FIRST_NAME') and (p.text or '').strip(): names.append(p.text.strip())
        except Exception as e: err=str(e)
    m0=len(rows)
    for _,p in peps.iterrows():
        for n in names[:4000]:
            sc=score_name_match(p['nombre_persona_input'],n)
            if classify_match(sc)!='discard': add(p,src,n,sc,path,url)
    bench.append({'source':src,'source_type':'sanctions_list','attempted':True,'success':ok and err=='','rows_downloaded_or_loaded':len(names),'rows_parsed':len(names),'matches_found':len(rows)-m0,'unique_people_with_hits':len(set([r['seed_id'] for r in rows[m0:]])),'runtime_seconds':time.perf_counter()-t0,'error_message':err,'notes':'Cache enabled'})
ev=pd.DataFrame(rows,columns=EVID_COLS); save_csv(ev,'notebooks/output/01_sanciones_ofac_onu_evidence.csv')
ps=ev.groupby(['seed_id','nombre_persona_input'],as_index=False).size().rename(columns={'size':'total_signals'}) if len(ev) else pd.DataFrame(columns=['seed_id','nombre_persona_input','total_signals'])
save_csv(ps,'notebooks/output/01_sanciones_ofac_onu_person_summary.csv'); save_csv(pd.DataFrame(bench),'notebooks/output/01_benchmark_sanciones.csv')
